In [1]:
import warnings
from collections import defaultdict

import tensordict
import torch
import torch.nn.functional as F
from env.env import Snapshot, load_cbor
from sklearn.model_selection import train_test_split
from tensordict import TensorDictBase, TensorDict
from tensordict.nn import TensorDictModule, CompositeDistribution, set_interaction_type, InteractionType
from torch import distributions as d
from torch import multiprocessing, nn
from torch.nn.functional import one_hot
from torch.utils.data import Dataset, DataLoader
from torchrl.collectors import Collector
from torchrl.data import Composite, MultiOneHot, UnboundedContinuous, ReplayBuffer, LazyTensorStorage, \
    SamplerWithoutReplacement
from torchrl.data.tensor_specs import BoundedDiscrete
from torchrl.envs import (Compose, DoubleToFloat, StepCounter,
                          TransformedEnv, EnvBase, check_env_specs, RewardScaling)
from torchrl.modules import ValueOperator, ProbabilisticActor
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

warnings.filterwarnings("ignore")

from env import Environment


In [2]:
is_fork = multiprocessing.get_start_method() == "fork"
device = torch.device("cpu")
lr = 3e-4  # Increased for better gradient flow in PPO
max_grad_norm = 0.5

In [3]:
sub_batch_size = 32
num_epochs = 4
clip_epsilon = 0.15
gamma = 0.995
lmbda = 0.95
entropy_eps = 0.01  # Reduced from 0.1 - was causing instability
bc_lr = 5e-4
bc_max_grad_norm = 0.5
head_weight_clip = 0.25
tensordict.nn.set_composite_lp_aggregate(False).set()
torch.autograd.set_detect_anomaly(False)  # Disable to avoid overhead, but keep NaN checking

def sanitize_state(x: torch.Tensor) -> torch.Tensor:
    # Keep board/piece encoding in expected numeric bounds.
    return torch.clamp(torch.nan_to_num(x, nan=0.0, posinf=8.0, neginf=0.0), 0.0, 8.0)

def sanitize_logits(logits: torch.Tensor, min_logit: float = -8.0) -> torch.Tensor:
    logits = torch.nan_to_num(logits, nan=min_logit, posinf=0.0, neginf=min_logit)
    logits = logits - logits.detach().amax(dim=-1, keepdim=True)
    return torch.clamp(logits, min=min_logit, max=0.0)

def is_finite_tensor(x: torch.Tensor) -> bool:
    return torch.isfinite(x).all().item()

In [4]:
frames_per_batch = 1000
# For a complete training, bring the number of frames up to 1M
total_frames = 50_000

In [5]:
class TetrisEnvWrapper(EnvBase):
    metadata = {"render.modes": ["human"]}

    def _make_spec(self):
        self.observation_spec = Composite(
            state=BoundedDiscrete(low=0, high=8, shape=726, dtype=torch.float32)
        )
        self.action_spec = MultiOneHot((3, 3, 4))
        self.reward_spec = UnboundedContinuous()

    def _step(self, tensordict: TensorDictBase) -> TensorDictBase:
        action = tensordict["action"]
        action_one_hot = torch.cat([
            one_hot(action["movement"], 3),
            one_hot(action["rotation"], 3),
            one_hot(action["additional"], 4)
        ], -1) if isinstance(action, TensorDictBase) else action
        snap: Snapshot = self.env.step(action_one_hot.squeeze().tolist())
        state = sanitize_state(torch.tensor(snap.state(), dtype=torch.float32))
        reward = torch.nan_to_num(torch.tensor(snap.reward(), dtype=torch.float32), nan=0.0, posinf=100.0, neginf=-100.0)
        reward = torch.clamp(reward, -100.0, 100.0)
        out = TensorDict({
            "state": state,
            "reward": reward.unsqueeze(-1),
            "done": torch.tensor(snap.done(), dtype=torch.bool).unsqueeze(-1),
        }, tensordict.shape)
        return out

    def _reset(self, tensordict: TensorDictBase, **kwargs) -> TensorDictBase:
        self.env.reset()
        out = TensorDict({
            "state": sanitize_state(torch.tensor(self.env.state(), dtype=torch.float32)),
        }, batch_size=self.batch_size)
        return out

    def _set_seed(self, seed: int | None) -> None:
        # ignore for now
        pass

    def __init__(self, env: Environment, device: torch.device):
        super().__init__(device=device)
        self.env = env
        self._make_spec()


In [6]:
base_env = TetrisEnvWrapper(Environment(), device=device)
env = TransformedEnv(
    base_env,
    Compose([
        DoubleToFloat(),
        StepCounter(),
        RewardScaling(loc=-0.05, scale=0.2),
    ])
)

check_env_specs(env)

2026-04-09 09:39:07,340 [torchrl][INFO]    check_env_specs succeeded! [END]


In [7]:
class BaseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.spec = env.output_spec
        self.convnet = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding='same'),
            nn.ELU(),
            nn.Dropout(0.1),
            nn.Conv2d(8, 16, kernel_size=3, padding='same'),
            nn.ELU(),
            nn.Dropout(0.1),
            nn.Conv2d(16, 16, kernel_size=3, padding='same'),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.1),
            nn.Conv2d(16, 16, kernel_size=3, padding='same'),
            nn.LayerNorm([16, 12, 5]),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(3, 1), stride=(3, 1)),
            nn.Dropout(0.1),
            nn.Flatten(),
        )
        self.piecenet = nn.Sequential(
            nn.Linear(8 * (1 + 5), 32),
            nn.ELU(),
            nn.Linear(32, 16),
            nn.ELU(),
            nn.Linear(16, 16),
            nn.LayerNorm(16),
            nn.ELU()
        )
        self.combnet0 = nn.Sequential(
            nn.Linear(336, 128),
            nn.ELU(),
        )
        self.combnet1 = nn.Sequential(
            nn.Linear(128, 128),
            nn.ELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.ELU(),
        )
        self.combnet2 = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ELU(),
        )

    def forward(self, x: torch.Tensor):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        board, pieces = x.split([3 * 24 * 10, 1 + 5], 1)
        board = board.view(-1, 3, 24, 10)
        pieces = torch.nn.functional.one_hot(pieces.long(), 8).float().flatten(start_dim=1)
        b = self.convnet(board)
        p = self.piecenet(pieces)
        comb = torch.cat([b, p], dim=1)
        c = self.combnet0(comb)
        c = c + self.combnet1(c)
        c = self.combnet2(c)
        return c


In [8]:
td = env.reset()
td["action"] = env.action_spec.rand()
td = env.step(td)

state = td["next", "state"].unsqueeze(0)  # (1, 726)

model = BaseNetwork()
model.eval()
with torch.no_grad():
    out = model(state)

print("Output shape:", out.shape)
print("Output:", out)

Output shape: torch.Size([1, 64])
Output: tensor([[-0.6463,  0.0235,  0.9345,  0.7326, -0.3945,  0.2734, -0.1430, -0.0635,
          0.2426,  0.5142, -0.2525, -0.1840,  0.3276, -0.3992,  0.5309, -0.3374,
         -0.3099, -0.4121, -0.2437,  0.9321,  0.2199,  0.5363, -0.0416, -0.5836,
          0.2243, -0.0465, -0.2439,  0.2191,  0.2226,  0.3657, -0.2085,  0.3794,
          0.4436,  1.5198, -0.5968, -0.4155, -0.3531, -0.3812,  0.3235, -0.1142,
          0.9013,  0.6897,  0.3071,  0.0459,  0.4707,  0.1778,  0.1411, -0.2188,
         -0.3310,  0.4376, -0.0352, -0.1689,  0.4550, -0.6938,  0.3836, -0.1527,
          0.7983,  0.6886, -0.1127, -0.4008, -0.3399,  0.0030, -0.0901, -0.6153]])


In [9]:
class ActorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = BaseNetwork()
        self.head_0 = nn.Linear(64, 3)  # movement
        torch.nn.init.xavier_uniform_(self.head_0.weight, gain=0.001)
        torch.nn.init.zeros_(self.head_0.bias)
        self.head_1 = nn.Linear(64, 3)  # rotation
        torch.nn.init.xavier_uniform_(self.head_1.weight, gain=0.001)
        torch.nn.init.zeros_(self.head_1.bias)
        self.head_2 = nn.Linear(64, 4)  # additional action
        torch.nn.init.xavier_uniform_(self.head_2.weight, gain=0.001)
        torch.nn.init.zeros_(self.head_2.bias)

    def forward(self, state):
        state = sanitize_state(state)
        features = torch.nan_to_num(self.base(state), nan=0.0, posinf=1e3, neginf=-1e3)
        # Compute logits with careful numerical stability
        l0 = self.head_0(features)
        l1 = self.head_1(features)
        l2 = self.head_2(features)

        l0 = sanitize_logits(l0)
        l1 = sanitize_logits(l1)
        l2 = sanitize_logits(l2)

        return l0, l1, l2


class CriticNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = BaseNetwork()
        self.head = nn.Linear(64, 1)

    def forward(self, state):
        state = sanitize_state(state)
        features = torch.nan_to_num(self.base(state), nan=0.0, posinf=1e3, neginf=-1e3)
        value = self.head(features)
        return torch.clamp(torch.nan_to_num(value, nan=0.0, posinf=100.0, neginf=-100.0), -100.0, 100.0)

In [10]:
actor_module = TensorDictModule(
    ActorNet(),
    in_keys=["state"],
    out_keys=[
        ("params", "movement", "logits"),
        ("params", "rotation", "logits"),
        ("params", "additional", "logits"),
    ],
)

actor = ProbabilisticActor(
    module=actor_module,
    in_keys=["params"],
    out_keys=[
        ("action", "movement"),
        ("action", "rotation"),
        ("action", "additional"),
    ],
    distribution_class=CompositeDistribution,
    distribution_kwargs={
        "distribution_map": {
            "movement": d.Categorical,
            "rotation": d.Categorical,
            "additional": d.Categorical,
        },
        "name_map": {
            "movement": ("action", "movement"),
            "rotation": ("action", "rotation"),
            "additional": ("action", "additional"),
        },
    },
    return_log_prob=True,
)

# Add a wrapper to catch inf/nan in probabilities
original_actor_forward = actor.forward
def actor_forward_safe(td):
    if "state" in td.keys():
        td["state"] = sanitize_state(td["state"])
    try:
        result = original_actor_forward(td)
        # Check logits for inf/nan
        for key in [("params", "movement", "logits"), ("params", "rotation", "logits"), ("params", "additional", "logits")]:
            if key in td:
                logits = td[key]
                if torch.isnan(logits).any() or torch.isinf(logits).any():
                    print(f"WARNING: inf/nan in logits {key}: min={logits.min()}, max={logits.max()}")
                    # Replace with safe values
                    td[key] = sanitize_logits(logits)
        return result
    except RuntimeError as e:
        if "categorical" in str(e).lower() or "probability" in str(e).lower():
            print(f"CRITICAL: {e}")
            print(f"Logits stats:")
            for key in [("params", "movement", "logits"), ("params", "rotation", "logits"), ("params", "additional", "logits")]:
                if key in td:
                    logits = td[key]
                    print(f"  {key}: min={logits.min()}, max={logits.max()}, mean={logits.mean()}, any_nan={torch.isnan(logits).any()}, any_inf={torch.isinf(logits).any()}")
                    td[key] = sanitize_logits(logits)
            raise
        raise

actor.forward = actor_forward_safe

critic = ValueOperator(
    CriticNet(),
    in_keys=["state"],
    out_keys=["state_value"],
)

In [11]:
collector = Collector(
    env,
    actor,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
    device=device,
)

replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=frames_per_batch),
    sampler=SamplerWithoutReplacement(),
)

In [ ]:
advantage_module = GAE(
    gamma=gamma, lmbda=lmbda, value_network=critic, average_gae=True, device=device,
)

loss_module = ClipPPOLoss(
    actor_network=actor,
    critic_network=critic,
    clip_epsilon=clip_epsilon,
    entropy_bonus=bool(entropy_eps),
    entropy_coeff=entropy_eps,
    normalize_advantage=True,  # Normalize advantages for better numerical stability
)

optim = torch.optim.Adam(loss_module.parameters(), lr, weight_decay=1e-5)

In [13]:
frames = []
for i in range(8):
    with open(f"game_data_{i + 1}.cbor", "rb") as file:
        frames.extend(load_cbor(file.read()))

print(len(frames))


class TetrisDataset(Dataset):
    def __init__(self, frames):
        self.frames = frames

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        frame = self.frames[idx]
        return (
            torch.tensor(frame["state"]), torch.tensor(frame["movement"]), torch.tensor(frame["rotation"]),
            torch.tensor(frame["additional"])
        )


X_train, X_test = train_test_split(frames, test_size=0.15, random_state=42, shuffle=True)
train_dataset = TetrisDataset(X_train)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_dataset = TetrisDataset(X_test)
val_loader = DataLoader(val_dataset, batch_size=256)

104306


In [14]:
optimizer = torch.optim.Adam(actor_module.parameters(), lr=bc_lr, weight_decay=1e-4)
best_val_loss = float("inf")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.2,
    patience=3,  # wait 3 epochs before reducing
)

for epoch in range(20):
    epoch_loss = 0
    valid_train_batches = 0
    for states, mov, rot, add in train_loader:
        states = sanitize_state(states.to(device))
        mov, rot, add = mov.to(device).long(), rot.to(device).long(), add.to(device).long()
        if not is_finite_tensor(states):
            print("WARNING: non-finite BC states, skipping batch")
            continue

        td = TensorDict({"state": states}, batch_size=len(states))
        actor_module(td)

        logits_mov = td["params", "movement", "logits"]  # (B, 3)
        logits_rot = td["params", "rotation", "logits"]  # (B, 3)
        logits_add = td["params", "additional", "logits"]  # (B, 4)

        # Safety check: ensure no inf/nan in logits
        if torch.isnan(logits_mov).any() or torch.isinf(logits_mov).any():
            print(f"WARNING: inf/nan in movement logits, skipping batch")
            continue
        if torch.isnan(logits_rot).any() or torch.isinf(logits_rot).any():
            print(f"WARNING: inf/nan in rotation logits, skipping batch")
            continue
        if torch.isnan(logits_add).any() or torch.isinf(logits_add).any():
            print(f"WARNING: inf/nan in additional logits, skipping batch")
            continue

        loss = (
                F.cross_entropy(logits_mov, mov) +
                F.cross_entropy(logits_rot, rot) +
                F.cross_entropy(logits_add, add)
        )

        # Check for inf/nan in loss
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"WARNING: inf/nan in cross_entropy loss, skipping batch")
            continue

        epoch_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()

        # Check for inf/nan gradients
        has_nan_grad = False
        for param in actor_module.parameters():
            if param.grad is not None and (torch.isnan(param.grad).any() or torch.isinf(param.grad).any()):
                has_nan_grad = True
                break

        if has_nan_grad:
            print(f"WARNING: inf/nan in gradients, skipping update")
            optimizer.zero_grad()
            continue

        grad_norm = torch.nn.utils.clip_grad_norm_(actor_module.parameters(), bc_max_grad_norm)
        if not torch.isfinite(grad_norm):
            print("WARNING: non-finite BC grad norm, skipping update")
            optimizer.zero_grad()
            continue
        optimizer.step()

        # Keep actor heads from becoming overconfident before PPO starts.
        for head in [actor_module.module.head_0, actor_module.module.head_1, actor_module.module.head_2]:
            head.weight.data.clamp_(-head_weight_clip, head_weight_clip)
            head.bias.data.clamp_(-head_weight_clip, head_weight_clip)

        valid_train_batches += 1

    actor_module.eval()
    val_loss = 0.0
    valid_val_batches = 0
    with torch.no_grad():
        for states, mov, rot, add in val_loader:
            states = sanitize_state(states.to(device))
            mov, rot, add = mov.to(device).long(), rot.to(device).long(), add.to(device).long()
            td = TensorDict({"state": states}, batch_size=len(states))
            actor_module(td)
            logits_mov = td["params", "movement", "logits"]
            logits_rot = td["params", "rotation", "logits"]
            logits_add = td["params", "additional", "logits"]
            if not (is_finite_tensor(logits_mov) and is_finite_tensor(logits_rot) and is_finite_tensor(logits_add)):
                continue
            val_loss += (
                    F.cross_entropy(logits_mov, mov) +
                    F.cross_entropy(logits_rot, rot) +
                    F.cross_entropy(logits_add, add)
            ).item()
            valid_val_batches += 1
    actor_module.train()

    val_loss /= max(valid_val_batches, 1)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(actor_module.state_dict(), "am_pt.pt")

    avg_loss = epoch_loss / max(valid_train_batches, 1)
    print(f"Epoch {epoch}: loss={avg_loss:.4f}, val loss: {val_loss:.4f}")
    scheduler.step(avg_loss, epoch=epoch)

Epoch 0: loss=1.8453, val loss: 1.4957
Epoch 1: loss=1.4868, val loss: 1.4625
Epoch 2: loss=1.4307, val loss: 1.4000
Epoch 3: loss=1.3917, val loss: 1.3656
Epoch 4: loss=1.3662, val loss: 1.3218
Epoch 5: loss=1.3395, val loss: 1.3076


KeyboardInterrupt: 

In [ ]:
actor_module.load_state_dict(torch.load("am_pt.pt"))

In [ ]:
actor_module.module.head_0.weight

Parameter containing:
tensor([[ 0.0251,  0.0023,  0.0254,  0.0216,  0.0429,  0.0257, -0.0398,  0.0246,
          0.0238,  0.0255,  0.0231,  0.0056,  0.0652,  0.0981, -0.1042,  0.0006,
         -0.1089,  0.0222,  0.0265,  0.0249,  0.0238,  0.0206,  0.0014,  0.0098,
          0.0582,  0.0210,  0.0259,  0.0390,  0.0145,  0.0223, -0.0692,  0.0166,
         -0.0094, -0.0370, -0.0920, -0.0286,  0.0257,  0.0236, -0.0868,  0.0346,
          0.0148,  0.0233,  0.0214, -0.0517, -0.0490, -0.0205,  0.0382,  0.0221,
          0.0250, -0.0742,  0.0267,  0.0242, -0.0049, -0.0481,  0.0184,  0.0312,
         -0.0009,  0.0255,  0.0091,  0.0661,  0.0062, -0.0491, -0.0893,  0.0412],
        [ 0.0185,  0.0396,  0.0149,  0.0146, -0.0004,  0.0149, -0.0313,  0.0113,
          0.0211,  0.0170,  0.0208, -0.0017,  0.0296, -0.1115, -0.0462,  0.0367,
         -0.0319,  0.0436,  0.0198,  0.0193,  0.0190,  0.0204,  0.0639, -0.0008,
          0.0514,  0.0161,  0.0153,  0.0248,  0.0517,  0.0112,  0.0231,  0.0149,
     

In [ ]:
logs = defaultdict(list)
eval_str = ""
it = 0

# We iterate over the collector until it reaches the total number of frames it was
# designed to collect:
for i, tensordict_data in enumerate(collector):
    if "state" in tensordict_data.keys():
        tensordict_data["state"] = sanitize_state(tensordict_data["state"])
    if ("next", "state") in tensordict_data.keys(True):
        tensordict_data["next", "state"] = sanitize_state(tensordict_data["next", "state"])
    if ("next", "reward") in tensordict_data.keys(True):
        rew = torch.nan_to_num(tensordict_data["next", "reward"], nan=0.0, posinf=100.0, neginf=-100.0)
        tensordict_data["next", "reward"] = torch.clamp(rew, -100.0, 100.0)
    # we now have a batch of data to work with. Let's learn something from it.
    with torch.no_grad():
        advantage_module(tensordict_data)

         # Normalize advantages very aggressively to prevent explosion
        advantage = tensordict_data.get("advantage", None)
        if advantage is not None:
            # First clip extreme outliers
            advantage = torch.nan_to_num(advantage, nan=0.0, posinf=100.0, neginf=-100.0)
            advantage = torch.clamp(advantage, -100.0, 100.0)
            # Then normalize
            adv_mean = advantage.mean()
            adv_std = advantage.std() + 1e-8
            tensordict_data["advantage"] = (advantage - adv_mean) / adv_std
            # Final safety clip to reasonable range
            tensordict_data["advantage"] = torch.clamp(tensordict_data["advantage"], -5.0, 5.0)

    data_view = tensordict_data.reshape(-1)
    print("Max reward: ", tensordict_data["next", "reward"].max().item())

    # Additional safety check on logits that were generated during collection
    invalid_batch = False
    if ("params", "movement", "logits") in tensordict_data:
        for key in [("params", "movement", "logits"), ("params", "rotation", "logits"), ("params", "additional", "logits")]:
            logits = tensordict_data[key]
            if torch.isnan(logits).any() or torch.isinf(logits).any():
                print(f"WARNING: inf/nan detected in collected {key}")
                print(f"  Stats: min={logits.min()}, max={logits.max()}, mean={logits.mean()}")
                # Don't use this batch to avoid poisoning the replay buffer
                invalid_batch = True
                break
            tensordict_data[key] = sanitize_logits(logits)

    if "sample_log_prob" in tensordict_data.keys(True):
        slp = tensordict_data["sample_log_prob"]
        tensordict_data["sample_log_prob"] = torch.clamp(torch.nan_to_num(slp, nan=-8.0, posinf=0.0, neginf=-8.0), -8.0, 0.0)

    if invalid_batch:
        continue

    replay_buffer.extend(data_view.cpu())
    for _ in range(num_epochs):
        for f in range(frames_per_batch // sub_batch_size):
            subdata = replay_buffer.sample(sub_batch_size)
            if "state" in subdata.keys():
                subdata["state"] = sanitize_state(subdata["state"])
            if ("next", "state") in subdata.keys(True):
                subdata["next", "state"] = sanitize_state(subdata["next", "state"])
            if ("next", "reward") in subdata.keys(True):
                rew = torch.nan_to_num(subdata["next", "reward"], nan=0.0, posinf=100.0, neginf=-100.0)
                subdata["next", "reward"] = torch.clamp(rew, -100.0, 100.0)

            # Clip advantages to prevent extreme values
            advantage = subdata.get("advantage", None)
            if advantage is not None:
                # Aggressively clip advantages to prevent explosion
                advantage = torch.nan_to_num(advantage, nan=0.0, posinf=10.0, neginf=-10.0)
                subdata["advantage"] = torch.clamp(advantage, -10.0, 10.0)

            if "sample_log_prob" in subdata.keys(True):
                slp = subdata["sample_log_prob"]
                subdata["sample_log_prob"] = torch.clamp(torch.nan_to_num(slp, nan=-8.0, posinf=0.0, neginf=-8.0), -8.0, 0.0)

            loss_vals = loss_module(subdata.to(device))
            if (not torch.isfinite(loss_vals["loss_objective"])) or (not torch.isfinite(loss_vals["loss_critic"])) or (not torch.isfinite(loss_vals["loss_entropy"])):
                print(f"WARNING: non-finite PPO components: objective={loss_vals['loss_objective']}, critic={loss_vals['loss_critic']}, entropy={loss_vals['loss_entropy']}")
                continue
            loss_value = (
                    loss_vals["loss_objective"]
                    + loss_vals["loss_critic"]
                    + loss_vals["loss_entropy"]
            )

            # Check for NaN/Inf before backprop
            if torch.isnan(loss_value) or torch.isinf(loss_value):
                print(f"WARNING: NaN/Inf detected in loss! loss_objective={loss_vals['loss_objective']}, "
                      f"loss_critic={loss_vals['loss_critic']}, loss_entropy={loss_vals['loss_entropy']}")
                # Print debug info
                if advantage is not None:
                    print(f"  Advantage stats: min={advantage.min()}, max={advantage.max()}, mean={advantage.mean()}")
                continue

            optim.zero_grad()
            loss_value.backward()

            bad_grad = False
            for p in loss_module.parameters():
                if p.grad is not None and (not torch.isfinite(p.grad).all()):
                    bad_grad = True
                    break
            if bad_grad:
                print("WARNING: non-finite PPO gradients, skipping update")
                optim.zero_grad()
                continue

            grad_norm = torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_grad_norm)
            if not torch.isfinite(grad_norm):
                print("WARNING: non-finite PPO grad norm, skipping update")
                optim.zero_grad()
                continue
            optim.step()

    logs["reward"].append(tensordict_data["next", "reward"].mean().item())
    it += tensordict_data.numel()
    cum_reward_str = (
        f"it {it}, average reward={logs['reward'][-1]: 4.4f}"
    )
    logs["step_count"].append(tensordict_data["step_count"].max().item())
    stepcount_str = f"step count (max): {logs['step_count'][-1]}"
    if i % 5 == 0:
        with set_interaction_type(InteractionType.DETERMINISTIC), torch.no_grad():
            eval_rollout = env.rollout(1000, actor)
            td = eval_rollout[0]  # single step
            actor_module.eval()
            actor_module(td)  # run just the TensorDictModule part
            actor_module.train()

            print("movement probs: ", F.softmax(td[("params", "movement", "logits")], dim=-1))
            print("rotation probs: ", F.softmax(td[("params", "rotation", "logits")], dim=-1))
            print("additional probs:", F.softmax(td[("params", "additional", "logits")], dim=-1))

            logs["eval reward"].append(eval_rollout["next", "reward"].mean().item())
            logs["eval reward (sum)"].append(
                eval_rollout["next", "reward"].mean().item()
            )
            logs["eval step_count"].append(eval_rollout["step_count"].max().item())
            eval_str = (
                f"eval  reward: {logs['eval reward (sum)'][-1]: 4.4f} "
                f"eval step-count: {logs['eval step_count'][-1]}"
            )
            del eval_rollout

    print(", ".join([eval_str, cum_reward_str, stepcount_str]))


Max reward:  0.03199995681643486
movement probs:  tensor([[0.3526, 0.4364, 0.2109]])
rotation probs:  tensor([[3.3524e-04, 9.9933e-01, 3.3524e-04]])
additional probs: tensor([[1.3407e-01, 4.3268e-01, 4.3311e-01, 1.4529e-04]])
eval  reward: -0.0474 eval step-count: 683, it 1000, average reward=-0.0703, step count (max): 280
Max reward:  0.0379999540746212
eval  reward: -0.0474 eval step-count: 683, it 2000, average reward=-0.0950, step count (max): 228
Max reward:  0.019999969750642776
eval  reward: -0.0474 eval step-count: 683, it 3000, average reward=-0.0631, step count (max): 326
Max reward:  0.04099992290139198
eval  reward: -0.0474 eval step-count: 683, it 4000, average reward=-0.0882, step count (max): 343
Max reward:  0.049999941140413284
eval  reward: -0.0474 eval step-count: 683, it 5000, average reward=-0.1444, step count (max): 212
Max reward:  0.043999943882226944
movement probs:  tensor([[0.4451, 0.5009, 0.0539]])
rotation probs:  tensor([[3.3524e-04, 9.9933e-01, 3.3524e-04

In [ ]:
import time

actor.eval()

td = env.reset()
total_reward = 0.0
step = 0

with set_interaction_type(InteractionType.DETERMINISTIC), torch.no_grad():
    while True:
        # Get action probabilities for display
        actor(td)
        mov_probs = F.softmax(td["params", "movement", "logits"], dim=-1).squeeze()
        rot_probs = F.softmax(td["params", "rotation", "logits"], dim=-1).squeeze()
        add_probs = F.softmax(td["params", "additional", "logits"], dim=-1).squeeze()

        # Sample action and step
        td = env.step(td)
        reward = td["next", "reward"].item()
        done = td["next", "done"].item()
        total_reward += reward
        step += 1

        # Render board from state
        state = td["state"].squeeze().tolist()
        board = state[:3 * 24 * 10]  # first 3*24*10 elements are the board

        # Print board (channel 0 = board occupancy, channel 1 = current piece)
        print("\033[H\033[J", end="")  # clear terminal
        print(f"Step: {step}  Reward: {total_reward:.2f}")
        print(f"Move:   L={mov_probs[0]:.2f} R={mov_probs[1]:.2f} _={mov_probs[2]:.2f}")
        print(f"Rotate: CW={rot_probs[0]:.2f} CCW={rot_probs[1]:.2f} _={rot_probs[2]:.2f}")
        print(f"Extra:  HD={add_probs[0]:.2f} SD={add_probs[1]:.2f} Hold={add_probs[2]:.2f} _={add_probs[3]:.2f}")
        print("+" + "-" * 10 + "+")

        for row in range(4, 24):  # skip buffer rows at top
            line = "|"
            for col in range(10):
                occupied = board[row * 10 + col]  # channel 0: board
                piece = board[24 * 10 + row * 10 + col]  # channel 1: current piece
                ghost = board[2 * 24 * 10 + row * 10 + col]  # channel 2: ghost
                if piece > 0.5:
                    line += "█"
                elif ghost > 0.5:
                    line += "░"
                elif occupied > 0.5:
                    line += "█"
                else:
                    line += " "
            line += "|"
            print(line)
        print("+" + "-" * 10 + "+")

        if done:
            print(f"Game over at step {step}, total reward: {total_reward:.2f}")
            break

        # Step the tensordict forward
        td = td["next"]

        time.sleep(0.05)  # adjust for comfortable viewing speed

Step: 1  Reward: -0.11
Move:   L=0.40 R=0.47 _=0.13
Rotate: CW=0.00 CCW=1.00 _=0.00
Extra:  HD=0.06 SD=0.45 Hold=0.49 _=0.00
+----------+
|   ███    |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|   ░      |
|   ░░░    |
+----------+
Step: 2  Reward: -0.12
Move:   L=0.38 R=0.49 _=0.13
Rotate: CW=0.00 CCW=1.00 _=0.00
Extra:  HD=0.06 SD=0.45 Hold=0.49 _=0.00
+----------+
|    █     |
|   ███    |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|          |
|    ░     |
|   ░░░    |
+----------+
Step: 3  Reward: -0.13
Move:   L=0.34 R=0.48 _=0.19
Rotate: CW=0.00 CCW=1.00 _=0.00
Extra:  HD=0.09 SD=0.43 Hold=0.48 _=0.00
+----------+
|     █    |
|    ██    |
|     █    |
|

In [ ]:
actor_module.module.head_0.weight

Parameter containing:
tensor([[ 0.0122, -0.0117,  0.0119,  0.0084,  0.0389,  0.0127, -0.0537,  0.0116,
          0.0111,  0.0125,  0.0099,  0.0184,  0.0566,  0.0823, -0.1153, -0.0068,
         -0.1049,  0.0138,  0.0140,  0.0130,  0.0121,  0.0075, -0.0139,  0.0104,
          0.0467,  0.0078,  0.0127,  0.0269, -0.0039,  0.0096, -0.0629,  0.0260,
         -0.0231, -0.0297, -0.0909, -0.0212,  0.0141,  0.0102, -0.0824,  0.0260,
          0.0132,  0.0105,  0.0085, -0.0570, -0.0441, -0.0224,  0.0311,  0.0086,
          0.0132, -0.0745,  0.0145,  0.0115, -0.0267, -0.0443,  0.0044,  0.0215,
          0.0034,  0.0131, -0.0045,  0.0572,  0.0089, -0.0605, -0.0994,  0.0308],
        [ 0.0100,  0.0343,  0.0065,  0.0062,  0.0101,  0.0063, -0.0287,  0.0028,
          0.0126,  0.0084,  0.0125, -0.0048,  0.0393, -0.0949, -0.0357,  0.0405,
         -0.0270,  0.0400,  0.0110,  0.0104,  0.0102,  0.0123,  0.0660,  0.0029,
          0.0614,  0.0079,  0.0067,  0.0323,  0.0540,  0.0109,  0.0157,  0.0095,
     